# Patient Journey Analysis

1. **The Care Team**:
Who are the key providers actually caring for this patient
How often do they see each provider
what are the provider specialty
When was the most recent visit with each provider

2. **Diagnoses**:
what and frequency... help better understand these patients

3. Where are they getting infused (and where have they been infused)?

### Version Control

| Version | Description                                      |
|---------|--------------------------------------------------|
| v1.3 (11/17)      | Changed Dx, Tx time windows; deduped total_hcp_touchpoints |
| v1.2 (11/15)      | Get additional HCP information |
| v1.0 (11/12)      | Overview of patient journey, care team, diagnoses, and infusion locations. |



**TODO v1.4**: 

first_claim_date, primary_hcp (affiliated with patient), claim_vol_by_year (calendar)



### Patient-HCP Visit Summary View

This SQL code creates a temporary view `patient_hcp_visit_summary` to analyze key healthcare providers (HCPs) for MPSII patients based on diagnosis (Dx) and treatment (Tx) activity.

#### Summary:
- **Eligibility:** Patients with 2+ specified or unspecified MPSII diagnoses and qualifying treatment (Elaprase or infusion codes) between Aug 1, 2020 and Jul 31, 2025.
- **Claims Windows:** 
  - 5-year: Aug 1, 2020 – Jul 31, 2025
  - 3-year: Aug 1, 2022 – Jul 31, 2025
- **Provider Ranking:** For each patient, ranks HCPs by 3-year visit count (Dx + Tx), with tie-breakers on last visit date and NPI.
- **Visit Definition:** Visits include any claim (Dx or Tx) associated with the patient and provider, not limited to MPSII-relevant codes.
- **Outputs per Patient:**
  - First Dx HCP and visit stats (5-year)
  - First Tx HCP and visit stats (5-year)
  - Top 5 most-seen HCPs (3-year), with visit counts and last visit dates for both 3-year and 5-year windows

This view supports downstream analysis of care teams, provider specialties, and patient-HCP touchpoints.

In [0]:
%sql
WITH ranked_hcps AS (
    -- TX Window: Aug 2023 - Jul 2025 (2 years)
    -- DX Window: Aug 2020 - Jul 2025 (5 years back)
    
    SELECT DISTINCT 
        NPI, 
        SPECIALTY,
        PATIENT_ID, 
        Fill_date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        'TX' as PATIENT_TYPE,
        '1YR' as YEAR_PERIOD
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            Fill_Date,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            NPI, 
            SPECIALTY, 
            FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
            FROM (
                SELECT *, 
                       MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
                FROM (
                    SELECT *, 
                           RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            PRIORITY,
                            NO_OF_VISITS
                        FROM (
                            SELECT 
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,
                                CASE 
                                    WHEN SPECIALTY = 'Geneticist' THEN 1 
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                    WHEN SPECIALTY = 'PCP' THEN 4 
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6 
                                    ELSE 7
                                END AS PRIORITY,
                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM (
                                SELECT 
                                    tx_data.*,
                                    prov.HCO_PRIMARY_NPI,
                                    prov.PRIMARY_SPECIALTY,
                                    prov.SECONDARY_SPECIALTY,
                                    CASE 
                                        WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                                        WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                                        WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                                             prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                                        WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                                        WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                                        WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                                        WHEN tx_data.npi IS NULL THEN 'NA'
                                        ELSE 'Others'
                                    END AS SPECIALTY
                                FROM (
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                                        NDC11 AS CODE,
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE,
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND SERVICE_DATE BETWEEN '2019-08-01' AND '2023-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        PRESCRIBER_NPI AS NPI,
                                        NDC11 AS CODE,
                                        PHARMACY_EVENT_ID as EVENT_ID,
                                        FILL_DATE,
                                        NULL AS PLACE_OF_SERVICE,
                                        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                                        'PHARMACY_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_pharmacy_events
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND TRANSACTION_RESULT = 'PAID'
                                      AND FILL_DATE BETWEEN '2019-08-01' AND '2023-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        RENDERING_NPI AS NPI,
                                        PROCEDURE_CODE AS CODE,   
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE, 
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
                                      AND SERVICE_DATE BETWEEN '2019-08-01' AND '2023-07-31'
                                ) tx_data
                                LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
                                WHERE (
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                    )
                                    OR
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E763%'
                                                  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E763'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                        AND PATIENT_ID IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND SERVICE_DATE BETWEEN '2019-08-01' AND '2023-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND TRANSACTION_RESULT = 'PAID'
                                                  AND FILL_DATE BETWEEN '2019-08-01' AND '2023-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE PROCEDURE_CODE = 'J1743'
                                                  AND SERVICE_DATE BETWEEN '2019-08-01' AND '2023-07-31'
                                            )
                                        )
                                        AND PATIENT_ID NOT IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                                            )
                                        )
                                    )
                                )
                            )
                            GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                        )
                    )
                )
            ) 
        ) 
        WHERE HCP_RANK = 1
    )
    
    UNION ALL
    
    -- 2 YEAR (Aug 2023 - Jul 2024)
    -- TX Window: Aug 2018 - Jul 2022 (5 years through 2 years back)
    -- DX Window: Aug 2019 - Jul 2024 (5 years back)
    
    SELECT DISTINCT 
        NPI, 
        SPECIALTY,
        PATIENT_ID, 
        Fill_date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        'TX' as PATIENT_TYPE,
        '2YR' as YEAR_PERIOD
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            Fill_Date,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            NPI, 
            SPECIALTY, 
            FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
            FROM (
                SELECT *, 
                       MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
                FROM (
                    SELECT *, 
                           RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            PRIORITY,
                            NO_OF_VISITS
                        FROM (
                            SELECT 
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,
                                CASE 
                                    WHEN SPECIALTY = 'Geneticist' THEN 1 
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                    WHEN SPECIALTY = 'PCP' THEN 4 
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6 
                                    ELSE 7
                                END AS PRIORITY,
                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM (
                                SELECT 
                                    tx_data.*,
                                    prov.HCO_PRIMARY_NPI,
                                    prov.PRIMARY_SPECIALTY,
                                    prov.SECONDARY_SPECIALTY,
                                    CASE 
                                        WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                                        WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                                        WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                                             prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                                        WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                                        WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                                        WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                                        WHEN tx_data.npi IS NULL THEN 'NA'
                                        ELSE 'Others'
                                    END AS SPECIALTY
                                FROM (
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                                        NDC11 AS CODE,
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE,
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND SERVICE_DATE BETWEEN '2018-08-01' AND '2022-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        PRESCRIBER_NPI AS NPI,
                                        NDC11 AS CODE,
                                        PHARMACY_EVENT_ID as EVENT_ID,
                                        FILL_DATE,
                                        NULL AS PLACE_OF_SERVICE,
                                        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                                        'PHARMACY_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_pharmacy_events
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND TRANSACTION_RESULT = 'PAID'
                                      AND FILL_DATE BETWEEN '2018-08-01' AND '2022-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        RENDERING_NPI AS NPI,
                                        PROCEDURE_CODE AS CODE,   
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE, 
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
                                      AND SERVICE_DATE BETWEEN '2018-08-01' AND '2022-07-31'
                                ) tx_data
                                LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
                                WHERE (
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2019-08-01' AND '2024-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2019-08-01' AND '2024-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                    )
                                    OR
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E763%'
                                                  AND SERVICE_DATE BETWEEN '2019-08-01' AND '2024-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E763'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2019-08-01' AND '2024-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                        AND PATIENT_ID IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND SERVICE_DATE BETWEEN '2018-08-01' AND '2022-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND TRANSACTION_RESULT = 'PAID'
                                                  AND FILL_DATE BETWEEN '2018-08-01' AND '2022-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE PROCEDURE_CODE = 'J1743'
                                                  AND SERVICE_DATE BETWEEN '2018-08-01' AND '2022-07-31'
                                            )
                                        )
                                        AND PATIENT_ID NOT IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2019-08-01' AND '2024-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2019-08-01' AND '2024-07-31'
                                            )
                                        )
                                    )
                                )
                            )
                            GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                        )
                    )
                )
            ) 
        ) 
        WHERE HCP_RANK = 1
    )
    
    UNION ALL
    
    -- 3 YEAR (Aug 2022 - Jul 2023)
    -- TX Window: Aug 2017 - Jul 2021 (5 years through 2 years back)
    -- DX Window: Aug 2018 - Jul 2023 (5 years back)
    
    SELECT DISTINCT 
        NPI, 
        SPECIALTY,
        PATIENT_ID, 
        Fill_date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        'TX' as PATIENT_TYPE,
        '3YR' as YEAR_PERIOD
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            Fill_Date,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            NPI, 
            SPECIALTY, 
            FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
            FROM (
                SELECT *, 
                       MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
                FROM (
                    SELECT *, 
                           RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            PRIORITY,
                            NO_OF_VISITS
                        FROM (
                            SELECT 
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,
                                CASE 
                                    WHEN SPECIALTY = 'Geneticist' THEN 1 
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                    WHEN SPECIALTY = 'PCP' THEN 4 
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6 
                                    ELSE 7
                                END AS PRIORITY,
                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM (
                                SELECT 
                                    tx_data.*,
                                    prov.HCO_PRIMARY_NPI,
                                    prov.PRIMARY_SPECIALTY,
                                    prov.SECONDARY_SPECIALTY,
                                    CASE 
                                        WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                                        WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                                        WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                                             prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                                        WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                                        WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                                        WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                                        WHEN tx_data.npi IS NULL THEN 'NA'
                                        ELSE 'Others'
                                    END AS SPECIALTY
                                FROM (
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                                        NDC11 AS CODE,
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE,
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND SERVICE_DATE BETWEEN '2017-08-01' AND '2021-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        PRESCRIBER_NPI AS NPI,
                                        NDC11 AS CODE,
                                        PHARMACY_EVENT_ID as EVENT_ID,
                                        FILL_DATE,
                                        NULL AS PLACE_OF_SERVICE,
                                        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                                        'PHARMACY_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_pharmacy_events
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND TRANSACTION_RESULT = 'PAID'
                                      AND FILL_DATE BETWEEN '2017-08-01' AND '2021-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        RENDERING_NPI AS NPI,
                                        PROCEDURE_CODE AS CODE,   
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE, 
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
                                      AND SERVICE_DATE BETWEEN '2017-08-01' AND '2021-07-31'
                                ) tx_data
                                LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
                                WHERE (
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2018-08-01' AND '2023-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2018-08-01' AND '2023-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                    )
                                    OR
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E763%'
                                                  AND SERVICE_DATE BETWEEN '2018-08-01' AND '2023-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E763'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2018-08-01' AND '2023-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                        AND PATIENT_ID IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND SERVICE_DATE BETWEEN '2017-08-01' AND '2021-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND TRANSACTION_RESULT = 'PAID'
                                                  AND FILL_DATE BETWEEN '2017-08-01' AND '2021-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE PROCEDURE_CODE = 'J1743'
                                                  AND SERVICE_DATE BETWEEN '2017-08-01' AND '2021-07-31'
                                            )
                                        )
                                        AND PATIENT_ID NOT IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2018-08-01' AND '2023-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2018-08-01' AND '2023-07-31'
                                            )
                                        )
                                    )
                                )
                            )
                            GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                        )
                    )
                )
            ) 
        ) 
        WHERE HCP_RANK = 1
    )
    
    UNION ALL
    
    -- 4 YEAR (Aug 2021 - Jul 2022)
    -- TX Window: Aug 2016 - Jul 2020 (5 years through 2 years back)
    -- DX Window: Aug 2017 - Jul 2022 (5 years back)
    
    SELECT DISTINCT 
        NPI, 
        SPECIALTY,
        PATIENT_ID, 
        Fill_date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        'TX' as PATIENT_TYPE,
        '4YR' as YEAR_PERIOD
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            Fill_Date,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            NPI, 
            SPECIALTY, 
            FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
            FROM (
                SELECT *, 
                       MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
                FROM (
                    SELECT *, 
                           RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            PRIORITY,
                            NO_OF_VISITS
                        FROM (
                            SELECT 
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,
                                CASE 
                                    WHEN SPECIALTY = 'Geneticist' THEN 1 
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                    WHEN SPECIALTY = 'PCP' THEN 4 
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6 
                                    ELSE 7
                                END AS PRIORITY,
                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM (
                                SELECT 
                                    tx_data.*,
                                    prov.HCO_PRIMARY_NPI,
                                    prov.PRIMARY_SPECIALTY,
                                    prov.SECONDARY_SPECIALTY,
                                    CASE 
                                        WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                                        WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                                        WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                                             prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                                        WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                                        WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                                        WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                                        WHEN tx_data.npi IS NULL THEN 'NA'
                                        ELSE 'Others'
                                    END AS SPECIALTY
                                FROM (
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                                        NDC11 AS CODE,
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE,
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND SERVICE_DATE BETWEEN '2016-08-01' AND '2020-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        PRESCRIBER_NPI AS NPI,
                                        NDC11 AS CODE,
                                        PHARMACY_EVENT_ID as EVENT_ID,
                                        FILL_DATE,
                                        NULL AS PLACE_OF_SERVICE,
                                        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                                        'PHARMACY_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_pharmacy_events
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND TRANSACTION_RESULT = 'PAID'
                                      AND FILL_DATE BETWEEN '2016-08-01' AND '2020-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        RENDERING_NPI AS NPI,
                                        PROCEDURE_CODE AS CODE,   
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE, 
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
                                      AND SERVICE_DATE BETWEEN '2016-08-01' AND '2020-07-31'
                                ) tx_data
                                LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
                                WHERE (
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2017-08-01' AND '2022-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2017-08-01' AND '2022-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                    )
                                    OR
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E763%'
                                                  AND SERVICE_DATE BETWEEN '2017-08-01' AND '2022-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E763'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2017-08-01' AND '2022-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                        AND PATIENT_ID IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND SERVICE_DATE BETWEEN '2016-08-01' AND '2020-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND TRANSACTION_RESULT = 'PAID'
                                                  AND FILL_DATE BETWEEN '2016-08-01' AND '2020-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE PROCEDURE_CODE = 'J1743'
                                                  AND SERVICE_DATE BETWEEN '2016-08-01' AND '2020-07-31'
                                            )
                                        )
                                        AND PATIENT_ID NOT IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2017-08-01' AND '2022-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2017-08-01' AND '2022-07-31'
                                            )
                                        )
                                    )
                                )
                            )
                            GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                        )
                    )
                )
            ) 
        ) 
        WHERE HCP_RANK = 1
    )
    
    UNION ALL
    
    -- 5 YEAR (Aug 2020 - Jul 2021)
    -- TX Window: Aug 2015 - Jul 2019 (5 years through 2 years back)
    -- DX Window: Aug 2016 - Jul 2021 (5 years back)
    
    SELECT DISTINCT 
        NPI, 
        SPECIALTY,
        PATIENT_ID, 
        Fill_date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        'TX' as PATIENT_TYPE,
        '5YR' as YEAR_PERIOD
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            Fill_Date,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            NPI, 
            SPECIALTY, 
            FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
            FROM (
                SELECT *, 
                       MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
                FROM (
                    SELECT *, 
                           RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            PRIORITY,
                            NO_OF_VISITS
                        FROM (
                            SELECT 
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,
                                CASE 
                                    WHEN SPECIALTY = 'Geneticist' THEN 1 
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                    WHEN SPECIALTY = 'PCP' THEN 4 
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6 
                                    ELSE 7
                                END AS PRIORITY,
                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM (
                                SELECT 
                                    tx_data.*,
                                    prov.HCO_PRIMARY_NPI,
                                    prov.PRIMARY_SPECIALTY,
                                    prov.SECONDARY_SPECIALTY,
                                    CASE 
                                        WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                                        WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                                        WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                                             prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                                        WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                                        WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                                        WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                                        WHEN tx_data.npi IS NULL THEN 'NA'
                                        ELSE 'Others'
                                    END AS SPECIALTY
                                FROM (
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                                        NDC11 AS CODE,
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE,
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND SERVICE_DATE BETWEEN '2015-08-01' AND '2019-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        PRESCRIBER_NPI AS NPI,
                                        NDC11 AS CODE,
                                        PHARMACY_EVENT_ID as EVENT_ID,
                                        FILL_DATE,
                                        NULL AS PLACE_OF_SERVICE,
                                        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                                        'PHARMACY_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_pharmacy_events
                                    WHERE NDC11 IN ('54092070001','540920700')
                                      AND TRANSACTION_RESULT = 'PAID'
                                      AND FILL_DATE BETWEEN '2015-08-01' AND '2019-07-31'
    
                                    UNION
    
                                    SELECT DISTINCT 
                                        PATIENT_ID,
                                        RENDERING_NPI AS NPI,
                                        PROCEDURE_CODE AS CODE,   
                                        MEDICAL_EVENT_ID AS EVENT_ID,
                                        SERVICE_DATE AS FILL_DATE, 
                                        PLACE_OF_SERVICE,
                                        KH_PLAN_ID AS KH_PLAN,
                                        'MEDICAL_EVENTS' AS TABLE_NAME
                                    FROM com_edp_prd.com_raw.kom_medical_events 
                                    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
                                      AND SERVICE_DATE BETWEEN '2015-08-01' AND '2019-07-31'
                                ) tx_data
                                LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
                                WHERE (
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2016-08-01' AND '2021-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2016-08-01' AND '2021-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                    )
                                    OR
                                    tx_data.PATIENT_ID IN (
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                            FROM (
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    SERVICE_DATE AS FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E763%'
                                                  AND SERVICE_DATE BETWEEN '2016-08-01' AND '2021-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT 
                                                    PATIENT_ID,
                                                    FILL_DATE
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E763'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2016-08-01' AND '2021-07-31'
                                            )
                                            GROUP BY PATIENT_ID
                                        )
                                        WHERE NUMBER_OF_CLAIMS >= 2
                                        AND PATIENT_ID IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND SERVICE_DATE BETWEEN '2015-08-01' AND '2019-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE NDC11 IN ('54092070001','540920700')
                                                  AND TRANSACTION_RESULT = 'PAID'
                                                  AND FILL_DATE BETWEEN '2015-08-01' AND '2019-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events 
                                                WHERE PROCEDURE_CODE = 'J1743'
                                                  AND SERVICE_DATE BETWEEN '2015-08-01' AND '2019-07-31'
                                            )
                                        )
                                        AND PATIENT_ID NOT IN (
                                            SELECT DISTINCT PATIENT_ID 
                                            FROM (
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_medical_events
                                                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                                  AND SERVICE_DATE BETWEEN '2016-08-01' AND '2021-07-31'
    
                                                UNION
    
                                                SELECT DISTINCT PATIENT_ID
                                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                                WHERE DIAGNOSIS_CODE = 'E761'
                                                  AND TRANSACTION_STATUS = 'PAID'
                                                  AND FILL_DATE BETWEEN '2016-08-01' AND '2021-07-31'
                                            )
                                        )
                                    )
                                )
                            )
                            GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                        )
                    )
                )
            ) 
        ) 
        WHERE HCP_RANK = 1
    )
)

-- PIVOT TO ONE ROW PER PATIENT
SELECT 
    PATIENT_ID,
    
    -- 1 YEAR Columns
    MAX(CASE WHEN YEAR_PERIOD = '1YR' THEN NPI END) AS PRIMARY_HCP_1YR,
    # MAX(CASE WHEN YEAR_PERIOD = '1YR' THEN SPECIALTY END) AS SPECIALTY_1YR,
    # MAX(CASE WHEN YEAR_PERIOD = '1YR' THEN HCO_PRIMARY_NPI END) AS HCO_1YR,
    
    -- 2 YEAR Columns
    MAX(CASE WHEN YEAR_PERIOD = '2YR' THEN NPI END) AS PRIMARY_HCP_2YR,
    # MAX(CASE WHEN YEAR_PERIOD = '2YR' THEN SPECIALTY END) AS SPECIALTY_2YR,
    # MAX(CASE WHEN YEAR_PERIOD = '2YR' THEN HCO_PRIMARY_NPI END) AS HCO_2YR,
    
    -- 3 YEAR Columns
    MAX(CASE WHEN YEAR_PERIOD = '3YR' THEN NPI END) AS PRIMARY_HCP_3YR,
    # MAX(CASE WHEN YEAR_PERIOD = '3YR' THEN SPECIALTY END) AS SPECIALTY_3YR,
    # MAX(CASE WHEN YEAR_PERIOD = '3YR' THEN HCO_PRIMARY_NPI END) AS HCO_3YR,
    
    -- 4 YEAR Columns
    MAX(CASE WHEN YEAR_PERIOD = '4YR' THEN NPI END) AS PRIMARY_HCP_4YR,
    # MAX(CASE WHEN YEAR_PERIOD = '4YR' THEN SPECIALTY END) AS SPECIALTY_4YR,
    # MAX(CASE WHEN YEAR_PERIOD = '4YR' THEN HCO_PRIMARY_NPI END) AS HCO_4YR,
    
    -- 5 YEAR Columns
    MAX(CASE WHEN YEAR_PERIOD = '5YR' THEN NPI END) AS PRIMARY_HCP_5YR
    # MAX(CASE WHEN YEAR_PERIOD = '5YR' THEN SPECIALTY END) AS SPECIALTY_5YR,
    # MAX(CASE WHEN YEAR_PERIOD = '5YR' THEN HCO_PRIMARY_NPI END) AS HCO_5YR
    
FROM ranked_hcps
GROUP BY PATIENT_ID;

In [0]:
%sql
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
-- =============================================================================
-- Purpose: Identify key healthcare providers for MPSII patients (Elaprase GTM)
-- Windows:
--   5 yrs: 2020-08-01..2025-07-31
--   3 yrs: 2022-08-01..2025-07-31
-- Ranking basis:
--   Top 5 by 3-year visit count (Dx + Tx), tie-break: last visit (DESC), NPI (ASC)
-- Outputs include 3yr and 5yr counts for top-5, plus last_visit_3yr and last_visit_5yr
-- =============================================================================

CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ----------------------------------------------------------
-- Claims universes (5y, 3y) Dx + Tx, with NPIs
-- ----------------------------------------------------------
all_dx_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        NULL AS TREATMENT_CODE,
        NULL AS CODE_SOURCE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        NULL AS TREATMENT_CODE,
        NULL AS CODE_SOURCE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761','E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
all_tx_claims_5yr AS (
    -- Medical events with NDC (Elaprase)
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        NDC11 AS TREATMENT_CODE,
        'MEDICAL_NDC' AS CODE_SOURCE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION

    -- Pharmacy events with NDC (Elaprase)
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        NDC11 AS TREATMENT_CODE,
        'PHARMACY_NDC' AS CODE_SOURCE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION

    -- Medical procedures (J-codes / infusion codes etc.)
    SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE,
        PROCEDURE_CODE AS TREATMENT_CODE,
        'MEDICAL_PROC' AS CODE_SOURCE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
all_claims_5yr AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),
all_claims_3yr AS (
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761','E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2022-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2022-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),

-- ----------------------------------------------------------
-- First Dx / First Tx (5y)
-- ----------------------------------------------------------
first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),
first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),
first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),
first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),
first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),
first_tx_hcp_5yr_codes AS (
    SELECT
        fth.PATIENT_ID,
        fth.first_tx_hcp,
        -- all codes used on the first_tx_date for that HCP/patient
        concat_ws(',', collect_set(tx.TREATMENT_CODE)) AS first_tx_treatment_codes_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID   = tx.PATIENT_ID
     AND fth.first_tx_hcp = tx.NPI
     AND fth.first_tx_date = tx.FILL_DATE
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

-- ----------------------------------------------------------
-- Most-seen ranking (Top 5 by 3y) with 5y counts + 5y last-visit
-- ----------------------------------------------------------
most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

-- ----------------------------------------------------------
-- Historical First Dates (No Date Restrictions)
-- ----------------------------------------------------------
historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),
historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),
latest_claim AS (
    SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_claim_date
    FROM all_claims_5yr
    GROUP BY PATIENT_ID
),
-- Patient Level Information
patient_demographics AS (
    SELECT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER
    FROM com_edp_prd.com_raw.kom_patient_demographics
),
patient_geography AS (
    SELECT PATIENT_ID, patient_state
    FROM com_edp_prd.com_raw.kom_patient_geography
    WHERE VALID_TO_DATE > CURRENT_DATE()
)

-- -----------------------------
-- Final output
-- -----------------------------
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,   
    -- Historical First Dates
    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,
    lc.latest_claim_date,
    
    -- First Dx HCP
    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

-- First Tx HCP
fth.first_tx_hcp AS first_tx_hcp_5yr,
COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,
fthc.first_tx_treatment_codes_5yr AS first_tx_treatment_codes_5yr,

    -- Most Seen HCP #1..#5 (ranked by 3-year activity)
    
    -- #1 Most Seen
    ms1.NPI  AS most_seen_hcp1_3yr_ranked,
    COALESCE(ms1.visit_count_3yr, 0) AS most_seen_hcp1_visit_count_3yr,
    ms1.last_visit_3yr AS most_seen_hcp1_last_visit_3yr,
    
    -- #2 Most Seen
    ms2.NPI  AS most_seen_hcp2_3yr_ranked,
    COALESCE(ms2.visit_count_3yr, 0) AS most_seen_hcp2_visit_count_3yr,
    ms2.last_visit_3yr AS most_seen_hcp2_last_visit_3yr,

    -- #3 Most Seen
    ms3.NPI  AS most_seen_hcp3_3yr_ranked,
    COALESCE(ms3.visit_count_3yr, 0) AS most_seen_hcp3_visit_count_3yr,
    ms3.last_visit_3yr AS most_seen_hcp3_last_visit_3yr,
    
    -- #4 Most Seen
    ms4.NPI  AS most_seen_hcp4_3yr_ranked,
    COALESCE(ms4.visit_count_3yr, 0) AS most_seen_hcp4_visit_count_3yr,
    ms4.last_visit_3yr AS most_seen_hcp4_last_visit_3yr,
    
    -- #5 Most Seen
    ms5.NPI  AS most_seen_hcp5_3yr_ranked,
    COALESCE(ms5.visit_count_3yr, 0) AS most_seen_hcp5_visit_count_3yr,
    ms5.last_visit_3yr AS most_seen_hcp5_last_visit_3yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd          ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg             ON ep.PATIENT_ID = pg.PATIENT_ID
LEFT JOIN historical_first_dx hfdx         ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx         ON ep.PATIENT_ID = hftx.PATIENT_ID
LEFT JOIN latest_claim lc                  ON ep.PATIENT_ID = lc.PATIENT_ID
LEFT JOIN first_dx_hcp fdh                 ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs      ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth                 ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths      ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx   ON ep.PATIENT_ID = fthtx.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_codes fthc   ON ep.PATIENT_ID = fthc.PATIENT_ID
LEFT JOIN most_seen_combined_stats ms1     ON ep.PATIENT_ID = ms1.PATIENT_ID AND ms1.rank = 1
LEFT JOIN most_seen_combined_stats ms2     ON ep.PATIENT_ID = ms2.PATIENT_ID AND ms2.rank = 2
LEFT JOIN most_seen_combined_stats ms3     ON ep.PATIENT_ID = ms3.PATIENT_ID AND ms3.rank = 3
LEFT JOIN most_seen_combined_stats ms4     ON ep.PATIENT_ID = ms4.PATIENT_ID AND ms4.rank = 4
LEFT JOIN most_seen_combined_stats ms5     ON ep.PATIENT_ID = ms5.PATIENT_ID AND ms5.rank = 5
ORDER BY ep.PATIENT_ID;


In [0]:
%sql
SELECT *
FROM patient_hcp_visit_summary

In [0]:
%sql
SELECT * 
-- FROM patient_hcp_visit_summary
from com_edp_prd.cmpa_insights_internal_schema.patient360

In [0]:
%sql
-- Verify the data
SELECT COUNT(*) as total_patients_patient360 FROM com_edp_prd.cmpa_insights_internal_schema.patient360;

In [0]:
%sql
-- HCP-Level Aggregation (optimized with unique patient counts)
-- One pass over patient_hcp_visit_summary using explode + role pivot

WITH phvs AS (
  SELECT *
  FROM patient_hcp_visit_summary
),

-- Long form: (patient_id, npi, role)
role_long AS (
  SELECT PATIENT_ID, first_dx_hcp_5yr       AS NPI, 'FIRST_DX'  AS role FROM phvs WHERE first_dx_hcp_5yr IS NOT NULL
  UNION ALL
  SELECT PATIENT_ID, first_tx_hcp_5yr       AS NPI, 'FIRST_TX'  AS role FROM phvs WHERE first_tx_hcp_5yr IS NOT NULL
  UNION ALL
  SELECT PATIENT_ID, npi, 'TOP5_3YR' AS role
  FROM phvs
  LATERAL VIEW explode(array(
      most_seen_hcp1_3yr_ranked,
      most_seen_hcp2_3yr_ranked,
      most_seen_hcp3_3yr_ranked,
      most_seen_hcp4_3yr_ranked,
      most_seen_hcp5_3yr_ranked
  )) e AS npi
  WHERE npi IS NOT NULL
),

-- Count distinct patients per (NPI, role)
role_counts AS (
  SELECT NPI, role, COUNT(DISTINCT PATIENT_ID) AS cnt
  FROM role_long
  GROUP BY NPI, role
),

-- Pivot roles to columns
counts_pivot AS (
  SELECT
    NPI,
    MAX(CASE WHEN role = 'FIRST_DX'  THEN cnt END) AS FIRST_DX_COUNT,
    MAX(CASE WHEN role = 'FIRST_TX'  THEN cnt END) AS FIRST_TX_COUNT,
    MAX(CASE WHEN role = 'TOP5_3YR'  THEN cnt END) AS MOST_SEEN_TOP5_COUNT
  FROM role_counts
  GROUP BY NPI
),

-- Count unique patients per NPI across all roles
unique_patient_counts AS (
  SELECT NPI, COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENT_TOUCHPOINTS
  FROM role_long
  GROUP BY NPI
)

SELECT
  c.NPI,
  CONCAT_WS(' ', p.FIRST_NAME, p.LAST_NAME) AS HCP_NAME,
  p.FIRST_NAME,
  p.LAST_NAME,
  p.PRIMARY_SPECIALTY,
  p.ORGANIZATION_NAME,
  p.SECONDARY_SPECIALTY,
  CASE 
    WHEN p.PRIMARY_SPECIALTY LIKE '%Genetic%' OR p.SECONDARY_SPECIALTY LIKE '%Genetic%' THEN 'Geneticist'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Pediatrics%' THEN 'Pediatrician'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Psychiatry & Neurology%' 
      OR p.SECONDARY_SPECIALTY LIKE '%Neurodevelopmental Disabilities%' 
      OR p.PRIMARY_SPECIALTY LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Nurse Practitioner%' OR p.PRIMARY_SPECIALTY LIKE '%Physician Assistant%' THEN 'NPPA'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Internal Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Internal Medicine%' THEN 'PCP'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Family Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Family Medicine%' THEN 'PCP'
    ELSE 'Others'
  END AS SPECIALTY_CATEGORY,
  p.HCO_PRIMARY_NPI,
  COALESCE(c.FIRST_DX_COUNT, 0)        AS FIRST_DX_COUNT,
  COALESCE(c.FIRST_TX_COUNT, 0)        AS FIRST_TX_COUNT,
  COALESCE(c.MOST_SEEN_TOP5_COUNT, 0)  AS MOST_SEEN_TOP5_COUNT,
  COALESCE(upc.UNIQUE_PATIENT_TOUCHPOINTS, 0) AS UNIQUE_PATIENT_TOUCHPOINTS
FROM counts_pivot c
LEFT JOIN unique_patient_counts upc
  ON c.NPI = upc.NPI
LEFT JOIN com_edp_prd.com_raw.kom_providers p
  ON c.NPI = p.NPI
ORDER BY UNIQUE_PATIENT_TOUCHPOINTS DESC, FIRST_DX_COUNT DESC, FIRST_TX_COUNT DESC, MOST_SEEN_TOP5_COUNT DESC;

In [0]:
%sql
select * from com_edp_prd.com_raw.kom_providers
where npi in ('1235234535');

select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping limit 10;

In [0]:
%sql
-- HCP-Level Aggregation (optimized with unique patient counts)
-- One pass over patient_hcp_visit_summary using explode + role pivot

WITH phvs AS (
  SELECT *
  FROM patient_hcp_visit_summary
),

-- Long form: (patient_id, npi, role)
role_long AS (
  SELECT PATIENT_ID, first_dx_hcp_5yr       AS NPI, 'FIRST_DX'  AS role FROM phvs WHERE first_dx_hcp_5yr IS NOT NULL
  UNION ALL
  SELECT PATIENT_ID, first_tx_hcp_5yr       AS NPI, 'FIRST_TX'  AS role FROM phvs WHERE first_tx_hcp_5yr IS NOT NULL
  UNION ALL
  SELECT PATIENT_ID, npi, 'TOP5_3YR' AS role
  FROM phvs
  LATERAL VIEW explode(array(
      most_seen_hcp1_3yr_ranked,
      most_seen_hcp2_3yr_ranked,
      most_seen_hcp3_3yr_ranked,
      most_seen_hcp4_3yr_ranked,
      most_seen_hcp5_3yr_ranked
  )) e AS npi
  WHERE npi IS NOT NULL
),

-- Count distinct patients per (NPI, role)
role_counts AS (
  SELECT NPI, role, COUNT(DISTINCT PATIENT_ID) AS cnt
  FROM role_long
  GROUP BY NPI, role
),

-- Pivot roles to columns
counts_pivot AS (
  SELECT
    NPI,
    MAX(CASE WHEN role = 'FIRST_DX'  THEN cnt END) AS FIRST_DX_COUNT,
    MAX(CASE WHEN role = 'FIRST_TX'  THEN cnt END) AS FIRST_TX_COUNT,
    MAX(CASE WHEN role = 'TOP5_3YR'  THEN cnt END) AS MOST_SEEN_TOP5_COUNT
  FROM role_counts
  GROUP BY NPI
),

-- Count unique patients per NPI across all roles
unique_patient_counts AS (
  SELECT NPI, COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENT_TOUCHPOINTS
  FROM role_long
  GROUP BY NPI
)

SELECT
  c.NPI,
  CONCAT_WS(' ', p.FIRST_NAME, p.LAST_NAME) AS HCP_NAME,
  p.FIRST_NAME,
  p.LAST_NAME,
  p.PRIMARY_SPECIALTY,
  p.ORGANIZATION_NAME,
  p.SECONDARY_SPECIALTY,
  CASE 
    WHEN p.PRIMARY_SPECIALTY LIKE '%Genetic%' OR p.SECONDARY_SPECIALTY LIKE '%Genetic%' THEN 'Geneticist'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Pediatrics%' THEN 'Pediatrician'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Psychiatry & Neurology%' 
      OR p.SECONDARY_SPECIALTY LIKE '%Neurodevelopmental Disabilities%' 
      OR p.PRIMARY_SPECIALTY LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Nurse Practitioner%' OR p.PRIMARY_SPECIALTY LIKE '%Physician Assistant%' THEN 'NPPA'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Internal Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Internal Medicine%' THEN 'PCP'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Family Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Family Medicine%' THEN 'PCP'
    ELSE 'Others'
  END AS SPECIALTY_CATEGORY,
  p.HCO_PRIMARY_NPI,
  hco.PROVIDER_ZIP AS HCO_ZIP,
  terr.territory_name AS HCO_TERRITORY,
  COALESCE(c.FIRST_DX_COUNT, 0)        AS FIRST_DX_COUNT,
  COALESCE(c.FIRST_TX_COUNT, 0)        AS FIRST_TX_COUNT,
  COALESCE(c.MOST_SEEN_TOP5_COUNT, 0)  AS MOST_SEEN_TOP5_COUNT,
  COALESCE(upc.UNIQUE_PATIENT_TOUCHPOINTS, 0) AS UNIQUE_PATIENT_TOUCHPOINTS
FROM counts_pivot c
LEFT JOIN unique_patient_counts upc
  ON c.NPI = upc.NPI
LEFT JOIN com_edp_prd.com_raw.kom_providers p
  ON c.NPI = p.NPI
LEFT JOIN com_edp_prd.com_raw.kom_providers hco
  ON p.HCO_PRIMARY_NPI = hco.NPI
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping terr
  ON hco.PROVIDER_ZIP = terr.zipcode
ORDER BY UNIQUE_PATIENT_TOUCHPOINTS DESC, FIRST_DX_COUNT DESC, FIRST_TX_COUNT DESC, MOST_SEEN_TOP5_COUNT DESC;

In [0]:
%sql

-- ============================================
-- VALIDATION QUERY ON THE SAVED VIEW
-- ============================================

SELECT 
    'Total Patients in View' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary

UNION ALL

SELECT 
    'Patients with First Dx HCP' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary
WHERE first_dx_hcp_5yr IS NOT NULL

UNION ALL

SELECT 
    'Patients with First Tx HCP' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary
WHERE first_tx_hcp_5yr IS NOT NULL

UNION ALL

SELECT 
    'Patients with Most Seen HCP #1' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary
WHERE most_seen_hcp1_5yr IS NOT NULL

UNION ALL

SELECT 
    'Patients with Most Seen HCP #2' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary
WHERE most_seen_hcp2_5yr IS NOT NULL

UNION ALL

SELECT 
    'Patients with Most Seen HCP #3' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary
WHERE most_seen_hcp3_5yr IS NOT NULL

UNION ALL

SELECT 
    'Patients with NO HCP Information' AS metric,
    COUNT(DISTINCT PATIENT_ID) AS count
FROM patient_hcp_visit_summary
WHERE first_dx_hcp_5yr IS NULL 
  AND first_tx_hcp_5yr IS NULL 
  AND most_seen_hcp1_5yr IS NULL

ORDER BY 
    CASE 
        WHEN metric = 'Total Patients in View' THEN 1
        WHEN metric = 'Patients with First Dx HCP' THEN 2
        WHEN metric = 'Patients with First Tx HCP' THEN 3
        WHEN metric = 'Patients with Most Seen HCP #1' THEN 4
        WHEN metric = 'Patients with Most Seen HCP #2' THEN 5
        WHEN metric = 'Patients with Most Seen HCP #3' THEN 6
        WHEN metric = 'Patients with NO HCP Information' THEN 7
    END;

In [0]:
%sql
SELECT COUNT(PATIENT_ID) AS patientCount FROM PATIENTS_DATA_2DX_TX;

## Diagnosis: Fetch Rendering HCPs

Get the patients and their associated HCPs. This analysis performed only on medical_events data

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW 2Dx_Tx_MedicalEvents AS
(SELECT
  PATIENT_ID,
  SERVICE_DATE,
  PROCEDURE_CODE,
  NDC11,
  RENDERING_NPI,
  REFERRING_NPI,
  BILLING_NPI,
  PLACE_OF_SERVICE,
  KH_PLAN_ID,
  DIAGNOSIS_CODES
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PATIENT_ID IN 
  (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
ORDER BY PATIENT_ID ASC, SERVICE_DATE ASC);

SELECT * FROM `2dx_tx_medicalevents` LIMIT 10;

In [0]:
%sql
-- Get all rendering NPIs associated with MPS2 diagnosis for each patient
CREATE OR REPLACE TEMP VIEW ALL_MPS2_DX_NPIS AS
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI,
    MIN(SERVICE_DATE) AS FIRST_DX_DATE_WITH_NPI,
    MAX(SERVICE_DATE) AS LAST_DX_DATE_WITH_NPI,
    COUNT(DISTINCT SERVICE_DATE) AS NUM_DX_VISITS
FROM 2Dx_Tx_MedicalEvents
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')  -- E76.1: MPS2 Specified, E76.3: MPS2 Unspecified
GROUP BY PATIENT_ID, RENDERING_NPI
ORDER BY PATIENT_ID, FIRST_DX_DATE_WITH_NPI;

-- Join Komodo Data
CREATE OR REPLACE TEMP VIEW MPS2_DX_TX_CLAIMS AS
SELECT A.*, 
CONCAT(B.FIRST_NAME, ' ', B.LAST_NAME) AS HCP_NAME,
B.PRIMARY_SPECIALTY,
B.SECONDARY_SPECIALTY,
B.HCO_PRIMARY_NPI,
B.PROVIDER_ZIP AS HCO_ZIP
FROM ALL_MPS2_DX_NPIS A
LEFT JOIN com_edp_prd.com_raw.kom_providers B
ON A.RENDERING_NPI = B.NPI;

-- Show sample
SELECT * FROM MPS2_DX_TX_CLAIMS LIMIT 10;

In [0]:
%sql
-- Patient-Level MPSII Analysis with Dx and Tx Counts and Ranked NPIs
-- 5 Year Dx Window: 2020-04-01 to 2025-03-31
-- 2 Year Tx Window: 2023-04-01 to 2025-03-31
-- Eligible: 2+ Specified Dx + Tx OR 2+ Unspecified Dx + Elaprase Tx (incremental)

CREATE OR REPLACE TEMP VIEW patient_centric_data AS
WITH MPSII_Diagnoses_Specified AS (
    -- Get all E76.1 (Specified) diagnosis claims
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
    UNION ALL
    
    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),

Patients_2Dx_Specified AS (
    -- Filter for E76.1 patients with 2+ diagnosis dates
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    -- Get all E76.3 (Unspecified) diagnosis claims
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
    UNION ALL
    
    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),

Patients_2Dx_Unspecified AS (
    -- Filter for E76.3 patients with 2+ diagnosis dates
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    -- Get all treatment claims (broader codes)
    SELECT DISTINCT PATIENT_ID
    FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001', '540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001', '540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    ) t
),

MPSII_Treatment_Elaprase_Only AS (
    -- Get Elaprase-only treatment claims
    SELECT DISTINCT PATIENT_ID
    FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001', '540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001', '540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    ) t
),

Patients_2Dx_Specified_With_Treatment AS (
    -- E76.1 patients with 2+ Dx AND treatment
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t ON p.PATIENT_ID = t.PATIENT_ID
),

Patients_Incremental_Unspecified AS (
    -- E76.3 patients with 2+ Dx, Elaprase treatment, NOT already in specified group
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t ON p.PATIENT_ID = t.PATIENT_ID
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

eligible_patients AS (
    -- Final result: Specified + Incremental Unspecified (425 patients)
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

all_dx_claims AS (
    -- All diagnosis claims for eligible patients
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    
    UNION
    
    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),

all_tx_claims AS (
    -- All treatment claims for eligible patients
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    
    UNION
    
    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    
    UNION
    
    SELECT DISTINCT 
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),

patient_dx_counts AS (
    -- Total Dx counts per patient per NPI (only for non-null NPIs)
    SELECT 
        PATIENT_ID,
        NPI,
        COUNT(*) AS dx_count
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            NPI,
            FILL_DATE
        FROM all_dx_claims
        WHERE NPI IS NOT NULL
    ) distinct_dx
    GROUP BY PATIENT_ID, NPI
),

patient_tx_counts AS (
    -- Total Tx counts per patient per NPI (only for non-null NPIs)
    SELECT 
        PATIENT_ID,
        NPI,
        COUNT(*) AS tx_count
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            NPI,
            FILL_DATE
        FROM all_tx_claims
        WHERE NPI IS NOT NULL
    ) distinct_tx
    GROUP BY PATIENT_ID, NPI
),

patient_total_dx_all AS (
    -- Total Dx counts per patient (including NULL NPIs)
    SELECT 
        PATIENT_ID,
        COUNT(*) AS total_dx_count
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            FILL_DATE
        FROM all_dx_claims
    ) distinct_dx_all
    GROUP BY PATIENT_ID
),

patient_total_tx_all AS (
    -- Total Tx counts per patient (including NULL NPIs)
    SELECT 
        PATIENT_ID,
        COUNT(*) AS total_tx_count
    FROM (
        SELECT DISTINCT 
            PATIENT_ID,
            FILL_DATE
        FROM all_tx_claims
    ) distinct_tx_all
    GROUP BY PATIENT_ID
),

dx_ranked AS (
    -- Rank NPIs by Dx count for each patient
    SELECT 
        PATIENT_ID,
        NPI,
        dx_count,
        ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY dx_count DESC, NPI) AS dx_rank
    FROM patient_dx_counts
),

tx_ranked AS (
    -- Rank NPIs by Tx count for each patient
    SELECT 
        PATIENT_ID,
        NPI,
        tx_count,
        ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY tx_count DESC, NPI) AS tx_rank
    FROM patient_tx_counts
),

dx_dates_by_npi AS (
    -- First and last Dx dates per patient per NPI
    SELECT 
        PATIENT_ID,
        NPI,
        MIN(FILL_DATE) AS first_dx_date,
        MAX(FILL_DATE) AS last_dx_date
    FROM all_dx_claims
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

tx_dates_by_npi AS (
    -- First and last Tx dates per patient per NPI
    SELECT 
        PATIENT_ID,
        NPI,
        MIN(FILL_DATE) AS first_tx_date,
        MAX(FILL_DATE) AS last_tx_date
    FROM all_tx_claims
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

patient_totals AS (
    -- Total Dx and Tx per patient - include ALL eligible patients
    SELECT 
        ep.PATIENT_ID,
        COALESCE(dx_all.total_dx_count, 0) AS total_dx_count,
        COALESCE(tx_all.total_tx_count, 0) AS total_tx_count
    FROM eligible_patients ep
    LEFT JOIN patient_total_dx_all dx_all ON ep.PATIENT_ID = dx_all.PATIENT_ID
    LEFT JOIN patient_total_tx_all tx_all ON ep.PATIENT_ID = tx_all.PATIENT_ID
)

-- Final output with all columns
SELECT 
    pt.PATIENT_ID,
    pt.total_dx_count,
    pt.total_tx_count,
    
    -- Top Tx NPI
    tx1.NPI AS top_tx_npi,
    tx1.tx_count AS top_tx_count,
    tx_dates1.first_tx_date AS top_tx_first_date,
    tx_dates1.last_tx_date AS top_tx_last_date,
    
    -- Second Tx NPI
    tx2.NPI AS second_tx_npi,
    tx2.tx_count AS second_tx_count,
    tx_dates2.first_tx_date AS second_tx_first_date,
    tx_dates2.last_tx_date AS second_tx_last_date,
    
    -- Third Tx NPI
    tx3.NPI AS third_tx_npi,
    tx3.tx_count AS third_tx_count,
    tx_dates3.first_tx_date AS third_tx_first_date,
    tx_dates3.last_tx_date AS third_tx_last_date,
    
    -- Top Dx NPI
    dx1.NPI AS top_dx_npi,
    dx1.dx_count AS top_dx_count,
    dx_dates1.first_dx_date AS top_dx_first_date,
    dx_dates1.last_dx_date AS top_dx_last_date,
    
    -- Second Dx NPI
    dx2.NPI AS second_dx_npi,
    dx2.dx_count AS second_dx_count,
    dx_dates2.first_dx_date AS second_dx_first_date,
    dx_dates2.last_dx_date AS second_dx_last_date,
    
    -- Third Dx NPI
    dx3.NPI AS third_dx_npi,
    dx3.dx_count AS third_dx_count,
    dx_dates3.first_dx_date AS third_dx_first_date,
    dx_dates3.last_dx_date AS third_dx_last_date,
    
    -- Patient type classification
    CASE 
        WHEN pt.PATIENT_ID IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment) THEN 'Tx (2Dx E761 + Tx)'
        WHEN pt.PATIENT_ID IN (SELECT PATIENT_ID FROM Patients_Incremental_Unspecified) THEN 'Tx (2Dx E763 + Elaprase - Incremental)'
        ELSE 'Other'
    END AS patient_category

FROM patient_totals pt

-- Join with ranked Tx NPIs
LEFT JOIN tx_ranked tx1 ON pt.PATIENT_ID = tx1.PATIENT_ID AND tx1.tx_rank = 1
LEFT JOIN tx_ranked tx2 ON pt.PATIENT_ID = tx2.PATIENT_ID AND tx2.tx_rank = 2
LEFT JOIN tx_ranked tx3 ON pt.PATIENT_ID = tx3.PATIENT_ID AND tx3.tx_rank = 3

-- Join with Tx dates
LEFT JOIN tx_dates_by_npi tx_dates1 ON pt.PATIENT_ID = tx_dates1.PATIENT_ID AND tx1.NPI = tx_dates1.NPI
LEFT JOIN tx_dates_by_npi tx_dates2 ON pt.PATIENT_ID = tx_dates2.PATIENT_ID AND tx2.NPI = tx_dates2.NPI
LEFT JOIN tx_dates_by_npi tx_dates3 ON pt.PATIENT_ID = tx_dates3.PATIENT_ID AND tx3.NPI = tx_dates3.NPI

-- Join with ranked Dx NPIs
LEFT JOIN dx_ranked dx1 ON pt.PATIENT_ID = dx1.PATIENT_ID AND dx1.dx_rank = 1
LEFT JOIN dx_ranked dx2 ON pt.PATIENT_ID = dx2.PATIENT_ID AND dx2.dx_rank = 2
LEFT JOIN dx_ranked dx3 ON pt.PATIENT_ID = dx3.PATIENT_ID AND dx3.dx_rank = 3

-- Join with Dx dates
LEFT JOIN dx_dates_by_npi dx_dates1 ON pt.PATIENT_ID = dx_dates1.PATIENT_ID AND dx1.NPI = dx_dates1.NPI
LEFT JOIN dx_dates_by_npi dx_dates2 ON pt.PATIENT_ID = dx_dates2.PATIENT_ID AND dx2.NPI = dx_dates2.NPI
LEFT JOIN dx_dates_by_npi dx_dates3 ON pt.PATIENT_ID = dx_dates3.PATIENT_ID AND dx3.NPI = dx_dates3.NPI

ORDER BY pt.total_tx_count DESC, pt.total_dx_count DESC, pt.PATIENT_ID;

In [0]:
%sql
-- Extract all unique NPIs from patient_centric_data view and get provider information
-- This includes all NPIs from top/second/third Dx and Tx columns

WITH all_npis AS (
    -- Extract all unique NPIs from the patient_centric_data view
    SELECT DISTINCT top_tx_npi AS NPI FROM patient_centric_data WHERE top_tx_npi IS NOT NULL
    UNION
    SELECT DISTINCT second_tx_npi AS NPI FROM patient_centric_data WHERE second_tx_npi IS NOT NULL
    UNION
    SELECT DISTINCT third_tx_npi AS NPI FROM patient_centric_data WHERE third_tx_npi IS NOT NULL
    UNION
    SELECT DISTINCT top_dx_npi AS NPI FROM patient_centric_data WHERE top_dx_npi IS NOT NULL
    UNION
    SELECT DISTINCT second_dx_npi AS NPI FROM patient_centric_data WHERE second_dx_npi IS NOT NULL
    UNION
    SELECT DISTINCT third_dx_npi AS NPI FROM patient_centric_data WHERE third_dx_npi IS NOT NULL
)
SELECT 
    p.NPI,
    p.FIRST_NAME,
    p.LAST_NAME,
    p.HCO_PRIMARY_NPI,
    p.PRIMARY_SPECIALTY,
    p.SECONDARY_SPECIALTY,
    CASE 
        WHEN p.PRIMARY_SPECIALTY LIKE '%Genetic%' OR p.SECONDARY_SPECIALTY LIKE '%Genetic%' THEN 'Geneticist'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Pediatrics%' THEN 'Pediatrician'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Psychiatry & Neurology%' OR p.SECONDARY_SPECIALTY LIKE '%Neurodevelopmental Disabilities%' OR 
             p.PRIMARY_SPECIALTY LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Nurse Practitioner%' OR p.PRIMARY_SPECIALTY LIKE '%Physician Assistant%' THEN 'NPPA'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Internal Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Internal Medicine%' THEN 'PCP'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Family Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Family Medicine%' THEN 'PCP'
        WHEN p.NPI IS NULL THEN 'NA'
        ELSE 'Others'
    END AS SPECIALTY_CATEGORY
FROM all_npis a
LEFT JOIN com_edp_prd.com_raw.kom_providers p ON a.NPI = p.NPI
ORDER BY p.LAST_NAME, p.FIRST_NAME;

In [0]:
%sql
-- Extract all unique NPIs from patient_centric_data view and get provider information
-- Includes reason for inclusion based on highest precedence role

WITH npi_roles AS (
    -- Extract NPIs with their roles and precedence
    SELECT DISTINCT top_tx_npi AS NPI, 'First_Treater' AS ROLE, 1 AS PRECEDENCE 
    FROM patient_centric_data WHERE top_tx_npi IS NOT NULL
    UNION
    SELECT DISTINCT top_dx_npi AS NPI, 'First_Dx' AS ROLE, 2 AS PRECEDENCE 
    FROM patient_centric_data WHERE top_dx_npi IS NOT NULL
    UNION
    SELECT DISTINCT second_tx_npi AS NPI, 'Second_Treater' AS ROLE, 3 AS PRECEDENCE 
    FROM patient_centric_data WHERE second_tx_npi IS NOT NULL
    UNION
    SELECT DISTINCT second_dx_npi AS NPI, 'Second_Dx' AS ROLE, 4 AS PRECEDENCE 
    FROM patient_centric_data WHERE second_dx_npi IS NOT NULL
    UNION
    SELECT DISTINCT third_tx_npi AS NPI, 'Third_Treater' AS ROLE, 5 AS PRECEDENCE 
    FROM patient_centric_data WHERE third_tx_npi IS NOT NULL
    UNION
    SELECT DISTINCT third_dx_npi AS NPI, 'Third_Dx' AS ROLE, 6 AS PRECEDENCE 
    FROM patient_centric_data WHERE third_dx_npi IS NOT NULL
),
npi_highest_role AS (
    -- Get the highest precedence role for each NPI
    SELECT 
        NPI,
        ROLE AS INCLUSION_REASON,
        PRECEDENCE
    FROM (
        SELECT 
            NPI,
            ROLE,
            PRECEDENCE,
            ROW_NUMBER() OVER (PARTITION BY NPI ORDER BY PRECEDENCE ASC) AS rn
        FROM npi_roles
    ) ranked
    WHERE rn = 1
)
SELECT 
    p.NPI,
    p.FIRST_NAME,
    p.LAST_NAME,
    nr.INCLUSION_REASON,
    p.HCO_PRIMARY_NPI,
    p.PRIMARY_SPECIALTY,
    p.SECONDARY_SPECIALTY,
    CASE 
        WHEN p.PRIMARY_SPECIALTY LIKE '%Genetic%' OR p.SECONDARY_SPECIALTY LIKE '%Genetic%' THEN 'Geneticist'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Pediatrics%' THEN 'Pediatrician'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Psychiatry & Neurology%' OR p.SECONDARY_SPECIALTY LIKE '%Neurodevelopmental Disabilities%' OR 
             p.PRIMARY_SPECIALTY LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Nurse Practitioner%' OR p.PRIMARY_SPECIALTY LIKE '%Physician Assistant%' THEN 'NPPA'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Internal Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Internal Medicine%' THEN 'PCP'
        WHEN p.PRIMARY_SPECIALTY LIKE '%Family Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Family Medicine%' THEN 'PCP'
        WHEN p.NPI IS NULL THEN 'NA'
        ELSE 'Others'
    END AS SPECIALTY_CATEGORY
FROM npi_highest_role nr
LEFT JOIN com_edp_prd.com_raw.kom_providers p ON nr.NPI = p.NPI
ORDER BY nr.PRECEDENCE, p.LAST_NAME, p.FIRST_NAME;

Based on this table, come up with another logic that gives to recency, frequency and specialty to assing HCP to a patient


### Diagnosis: PATIENT_PRIMARY_HCP View

This SQL creates a temporary view `PATIENT_PRIMARY_HCP` to assign a primary healthcare provider (HCP) to each patient using three logic strategies:

1. **First Incidence HCP**: The provider associated with the patient's earliest diagnosis date, breaking ties by recency, visit volume, and NPI.
2. **Most DX Visits HCP**: The provider with the highest number of diagnosis visits, breaking ties by specialty priority, recency, and NPI.
3. **Specialty-Weighted/Recency/Volume HCP**: The provider ranked by specialty priority (genetic, neurology, pediatric, etc.), then by most recent visit, then by visit volume.

Specialty priority is assigned numerically (lower is better) based on keywords in provider specialties. The final output includes all three HCP assignments per patient, with fallback logic if the "most visits" provider is missing.

**Source Table:** `MPS2_DX_TX_CLAIMS`  
**Output Columns:**  
- `PATIENT_ID`
- `FIRST_INCIDENCE_HCP`, `FIRST_INCIDENCE_HCP_NAME`
- `MOST_DX_VISITS_HCP`, `MOST_DX_VISITS_HCP_NAME`
- `SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP`, `SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP_NAME`

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW PATIENT_PRIMARY_HCP AS
WITH base AS (
  SELECT
      PATIENT_ID,
      -- normalize NPI from float/scientific to string
      CAST(CAST(RENDERING_NPI AS DECIMAL(20,0)) AS STRING) AS NPI,
      to_date(FIRST_DX_DATE_WITH_NPI) AS FIRST_DX_DATE,
      to_date(LAST_DX_DATE_WITH_NPI)  AS LAST_DX_DATE,
      NUM_DX_VISITS,
      HCP_NAME,
      PRIMARY_SPECIALTY,
      SECONDARY_SPECIALTY
  FROM MPS2_DX_TX_CLAIMS
),
enriched AS (
  SELECT
      b.*,
      /* Specialty priority (lower is better) */
      CASE
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%genetic%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%genetic%'                THEN 1
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%psychiatry & neurology%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%neurodevelopmental%'
          OR lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%neurological surgery%'   THEN 2
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%pediatric%'              THEN 3
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%internal medicine%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%internal medicine%'
          OR lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%family medicine%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%family medicine%'        THEN 4
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%nurse practitioner%'
          OR lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%physician assistant%'    THEN 5
        WHEN NPI IS NULL                                                                THEN 7
        ELSE 6
      END AS PRIORITY
  FROM base b
),
/* 1) First-incidence: earliest first-date; ties → recency → volume → NPI, all null-safe */
first_incidence AS (
  SELECT PATIENT_ID, NPI AS FIRST_INCIDENCE_HCP, HCP_NAME AS FIRST_INCIDENCE_HCP_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               CASE WHEN FIRST_DX_DATE IS NULL THEN 1 ELSE 0 END,  -- real dates first
               FIRST_DX_DATE ASC,
               CASE WHEN LAST_DX_DATE  IS NULL THEN 1 ELSE 0 END,
               LAST_DX_DATE DESC,
               NUM_DX_VISITS DESC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
),
/* 2) Most DX visits: volume primary; ties → specialty → recency → NPI (guarantees a row) */
most_dx_visits AS (
  SELECT PATIENT_ID, NPI AS MOST_DX_VISITS_HCP, HCP_NAME AS MOST_DX_VISITS_HCP_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               COALESCE(NUM_DX_VISITS,0) DESC,
               PRIORITY ASC,                                    -- your “specialty condition”
               CASE WHEN LAST_DX_DATE IS NULL THEN 1 ELSE 0 END,
               LAST_DX_DATE DESC,                               -- your “recency condition”
               CASE WHEN FIRST_DX_DATE IS NULL THEN 1 ELSE 0 END,
               FIRST_DX_DATE ASC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
),
/* 3) Specialty-weighted + Recency + Volume: specialty → recency → volume */
srv AS (
  SELECT PATIENT_ID, NPI AS SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP,
         HCP_NAME AS SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               PRIORITY ASC,
               CASE WHEN LAST_DX_DATE IS NULL THEN 1 ELSE 0 END,
               LAST_DX_DATE DESC,
               COALESCE(NUM_DX_VISITS,0) DESC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
)

SELECT
  e.PATIENT_ID,
  fi.FIRST_INCIDENCE_HCP,
  /* If, for any reason, MOST is missing but SRV exists, fall back to SRV (should be rare) */
  COALESCE(mv.MOST_DX_VISITS_HCP, s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP) AS MOST_DX_VISITS_HCP,
  s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP,
  fi.FIRST_INCIDENCE_HCP_NAME,
  COALESCE(mv.MOST_DX_VISITS_HCP_NAME, s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP_NAME) AS MOST_DX_VISITS_HCP_NAME,
  s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP_NAME
FROM (SELECT DISTINCT PATIENT_ID FROM enriched) e
LEFT JOIN first_incidence fi ON e.PATIENT_ID = fi.PATIENT_ID
LEFT JOIN most_dx_visits mv  ON e.PATIENT_ID = mv.PATIENT_ID
LEFT JOIN srv s              ON e.PATIENT_ID = s.PATIENT_ID;



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW PATIENT_PRIMARY_TREATMENT_HCP AS
WITH treatment_base AS (
  -- Get all treatment claims with NPIs for patients in cohort
  SELECT
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS TREATMENT_DATE,
      NDC11 AS CODE,
      PLACE_OF_SERVICE
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
      AND (
          NDC11 IN ('54092070001', '540920700')
          OR PROCEDURE_CODE IN (
              '99601','99602','96365','96366','J1743','S9357','S9379',
              '38206','38230','38232','38240','38241','38242','38243','38250'
          )
      )
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
  
  UNION ALL
  
  -- Pharmacy events
  SELECT
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE AS TREATMENT_DATE,
      NDC11 AS CODE,
      NULL AS PLACE_OF_SERVICE
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
      AND NDC11 IN ('54092070001', '540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
treatment_aggregated AS (
  -- Aggregate treatment visits per patient-NPI
  SELECT
      PATIENT_ID,
      CAST(CAST(NPI AS DECIMAL(20,0)) AS STRING) AS NPI,
      MIN(TREATMENT_DATE) AS FIRST_TX_DATE,
      MAX(TREATMENT_DATE) AS LAST_TX_DATE,
      COUNT(DISTINCT TREATMENT_DATE) AS NUM_TX_VISITS
  FROM treatment_base
  WHERE NPI IS NOT NULL
  GROUP BY PATIENT_ID, NPI
),
treatment_with_provider AS (
  -- Join with provider info
  SELECT
      t.*,
      CONCAT(p.FIRST_NAME, ' ', p.LAST_NAME) AS HCP_NAME,
      p.PRIMARY_SPECIALTY,
      p.SECONDARY_SPECIALTY
  FROM treatment_aggregated t
  LEFT JOIN com_edp_prd.com_raw.kom_providers p
      ON t.NPI = p.NPI
),
enriched AS (
  SELECT
      t.*,
      /* Specialty priority (lower is better) */
      CASE
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%genetic%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%genetic%'                THEN 1
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%psychiatry & neurology%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%neurodevelopmental%'
          OR lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%neurological surgery%'   THEN 2
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%pediatric%'              THEN 3
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%internal medicine%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%internal medicine%'
          OR lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%family medicine%'
          OR lower(coalesce(SECONDARY_SPECIALTY,'')) LIKE '%family medicine%'        THEN 4
        WHEN lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%nurse practitioner%'
          OR lower(coalesce(PRIMARY_SPECIALTY,''))   LIKE '%physician assistant%'    THEN 5
        WHEN NPI IS NULL                                                                THEN 7
        ELSE 6
      END AS PRIORITY
  FROM treatment_with_provider t
),
/* 1) First Treater: earliest first treatment date; ties → recency → volume → NPI */
first_treater AS (
  SELECT PATIENT_ID, NPI AS FIRST_TREATER_HCP, HCP_NAME AS FIRST_TREATER_HCP_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               CASE WHEN FIRST_TX_DATE IS NULL THEN 1 ELSE 0 END,
               FIRST_TX_DATE ASC,
               CASE WHEN LAST_TX_DATE IS NULL THEN 1 ELSE 0 END,
               LAST_TX_DATE DESC,
               NUM_TX_VISITS DESC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
),
/* 2) Most Treatment Visits: volume primary; ties → specialty → recency → NPI */
most_tx_visits AS (
  SELECT PATIENT_ID, NPI AS MOST_TX_VISITS_HCP, HCP_NAME AS MOST_TX_VISITS_HCP_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               COALESCE(NUM_TX_VISITS,0) DESC,
               PRIORITY ASC,
               CASE WHEN LAST_TX_DATE IS NULL THEN 1 ELSE 0 END,
               LAST_TX_DATE DESC,
               CASE WHEN FIRST_TX_DATE IS NULL THEN 1 ELSE 0 END,
               FIRST_TX_DATE ASC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
),
/* 3) Recent Treater: most recent treatment date; ties → volume → specialty → NPI */
recent_treater AS (
  SELECT PATIENT_ID, NPI AS RECENT_TREATER_HCP, HCP_NAME AS RECENT_TREATER_HCP_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               CASE WHEN LAST_TX_DATE IS NULL THEN 1 ELSE 0 END,
               LAST_TX_DATE DESC,
               COALESCE(NUM_TX_VISITS,0) DESC,
               PRIORITY ASC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
),
/* 4) Specialty-weighted + Recency + Volume: specialty → recency → volume */
srv AS (
  SELECT PATIENT_ID, NPI AS SPECIALTY_WEIGHTED_RECENCY_VOLUME_TREATER,
         HCP_NAME AS SPECIALTY_WEIGHTED_RECENCY_VOLUME_TREATER_NAME
  FROM (
    SELECT e.*,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY
               PRIORITY ASC,
               CASE WHEN LAST_TX_DATE IS NULL THEN 1 ELSE 0 END,
               LAST_TX_DATE DESC,
               COALESCE(NUM_TX_VISITS,0) DESC,
               NPI
           ) AS rn
    FROM enriched e
  ) x WHERE rn = 1
)

SELECT
  e.PATIENT_ID,
  ft.FIRST_TREATER_HCP,
  ft.FIRST_TREATER_HCP_NAME,
  COALESCE(mv.MOST_TX_VISITS_HCP, s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_TREATER) AS MOST_TX_VISITS_HCP,
  COALESCE(mv.MOST_TX_VISITS_HCP_NAME, s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_TREATER_NAME) AS MOST_TX_VISITS_HCP_NAME,
  rt.RECENT_TREATER_HCP,
  rt.RECENT_TREATER_HCP_NAME,
  s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_TREATER,
  s.SPECIALTY_WEIGHTED_RECENCY_VOLUME_TREATER_NAME
FROM (SELECT DISTINCT PATIENT_ID FROM enriched) e
LEFT JOIN first_treater ft ON e.PATIENT_ID = ft.PATIENT_ID
LEFT JOIN most_tx_visits mv ON e.PATIENT_ID = mv.PATIENT_ID
LEFT JOIN recent_treater rt ON e.PATIENT_ID = rt.PATIENT_ID
LEFT JOIN srv s ON e.PATIENT_ID = s.PATIENT_ID;


-- Show sample
SELECT * FROM PATIENT_PRIMARY_TREATMENT_HCP LIMIT 100;

In [0]:
%sql
-- Sample
SELECT DISTINCT FIRST_INCIDENCE_HCP FROM PATIENT_PRIMARY_HCP
UNION
SELECT DISTINCT MOST_DX_VISITS_HCP FROM PATIENT_PRIMARY_HCP
UNION
SELECT DISTINCT SPECIALTY_WEIGHTED_RECENCY_VOLUME_HCP FROM PATIENT_PRIMARY_HCP;


**TODO**:
- Quantify the new HCPs (# of patients first dx, # of patients most frequent dx, # of patients first tx, # of patients most frequent tx)
- Compare this against the new TL
- For each patient, not including the HCPs we've identified, identify the other T3 HCPs (based on the frequency of Dx, Tx HCP touchpoints - last 1 year) with the most touched (# of TPs, Specialty)
- Bring in Specialty to the tables
- Bring in HCP, HCO info and include Territory
- Cross validate this against Jess' email files
- Validate the patient counts (401), get a sign-off on the filtering
- Anytime you're identifying a HCP, add the latest visit date for each of these HCPs at a patient level


## S2: Diagnostic Frequency

TODO: Extend the diagnostic frequency to the last 5 years

In [0]:
%sql
-- Get MPS2 diagnostic frequency at quarterly level for last 2 years
-- For patients in PATIENTS_DATA_2DX_TX cohort

WITH MPS2_Diagnosis_Claims AS (
    -- Get all MPS2 diagnosis claims from medical events
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DIAGNOSIS_DATE,
        'E761' AS DIAGNOSIS_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
        AND DIAGNOSIS_CODES LIKE '%E761%'  -- E76.1: MPS2 Specified
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
    UNION ALL
    
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DIAGNOSIS_DATE,
        'E763' AS DIAGNOSIS_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
        AND DIAGNOSIS_CODES LIKE '%E763%'  -- E76.3: MPS2 Unspecified
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
    UNION ALL
    
    -- Get all MPS2 diagnosis claims from pharmacy events
    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE AS DIAGNOSIS_DATE,
        DIAGNOSIS_CODE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
        AND DIAGNOSIS_CODE IN ('E761', 'E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Quarterly_Diagnosis AS (
    SELECT 
        PATIENT_ID,
        YEAR(DIAGNOSIS_DATE) AS YEAR,
        QUARTER(DIAGNOSIS_DATE) AS QUARTER,
        DATE_TRUNC('QUARTER', DIAGNOSIS_DATE) AS QUARTER_START_DATE,
        COUNT(DISTINCT DIAGNOSIS_DATE) AS DIAGNOSIS_FREQUENCY
    FROM MPS2_Diagnosis_Claims
    GROUP BY 
        PATIENT_ID, 
        YEAR(DIAGNOSIS_DATE), 
        QUARTER(DIAGNOSIS_DATE),
        DATE_TRUNC('QUARTER', DIAGNOSIS_DATE)
)
SELECT 
    PATIENT_ID,
    CONCAT(YEAR,'-','Q',QUARTER) AS QUARTER,
    DIAGNOSIS_FREQUENCY
FROM Quarterly_Diagnosis
ORDER BY PATIENT_ID ASC, YEAR ASC, QUARTER ASC;

## S3: Where are the patients getting infused?

Where are the patients getting infused?

-- Last 12 therapies
-- Do a percentage of the whole - office % vs home %
-- For plotting: aggregated at a monthly level

In [0]:
%sql
-- Get all place of service records for treatment patients with timestamps
-- Based on PATIENTS_DATA_2DX_TX cohort

WITH Treatment_Claims_With_POS AS (
    -- Get all treatment claims with place of service from medical events
    SELECT DISTINCT 
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM PATIENTS_DATA_2DX_TX)
        AND (
            NDC11 IN ('54092070001','540920700')  -- Elaprase NDCs
            OR PROCEDURE_CODE IN (
                '99601',   -- Home infusion/specialty drug administration
                '99602',   -- Home infusion/specialty drug administration
                '96365',   -- Intravenous infusion, initial
                '96366',   -- Intravenous infusion, additional hour
                'J1743',   -- Elaprase (idursulfase) injection
                'S9357',   -- Home infusion therapy
                'S9379',   -- Home infusion therapy
                '38206',   -- Blood-derived hematopoietic progenitor cell harvesting; autologous
                '38230',   -- Bone marrow harvesting; allogeneic
                '38232',   -- Bone marrow harvesting; autologous
                '38240',   -- Hematopoietic progenitor cell (HPC); allogeneic transplantation
                '38241',   -- Hematopoietic progenitor cell (HPC); autologous transplantation
                '38242',   -- Allogeneic lymphocyte infusions
                '38243',   -- Hematopoietic progenitor cell (HPC); HPC boost
                '38250'    -- Bone marrow or blood-derived peripheral stem cell transplantation
            )
        )
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        AND PLACE_OF_SERVICE IS NOT NULL
)
-- Return all place of service records
SELECT 
    PATIENT_ID,
    FILL_DATE AS DATE,
    description AS POS
FROM Treatment_Claims_With_POS A
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description B
ON A.place_of_service = B.code 
ORDER BY PATIENT_ID ASC, POS ASC, DATE ASC;

## Appendix

In [0]:
%sql
-- Create table with MPSII patients with 2+ Dx AND treatment
-- Dx window: 5 years (E76.1 Specified + E76.3 Unspecified incremental), Tx window: 2 years

CREATE OR REPLACE TEMP VIEW PATIENTS_DATA_2DX_TX AS
WITH MPSII_Diagnoses_Specified AS (
    -- Get all E76.1 (Specified) diagnosis claims
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'  -- E76.1: Mucopolysaccharidosis, type II (MPSII Specified)
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
    UNION ALL
    
    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'  -- E76.1: Mucopolysaccharidosis, type II (MPSII Specified)
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Specified AS (
    -- Filter for E76.1 patients with 2+ diagnosis dates
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    -- Get all E76.3 (Unspecified) diagnosis claims
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'  -- E76.3: Mucopolysaccharidosis, unspecified
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    
    UNION ALL
    
    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'  -- E76.3: Mucopolysaccharidosis, unspecified
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Unspecified AS (
    -- Filter for E76.3 patients with 2+ diagnosis dates
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    -- Get all treatment claims (broader codes)
    SELECT DISTINCT PATIENT_ID
    FROM (
        -- Elaprase NDCs from medical events
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN (
            '54092070001',  -- Elaprase (idursulfase)
            '540920700'     -- Elaprase (idursulfase)
        )
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        -- Elaprase NDCs from pharmacy
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN (
            '54092070001',  -- Elaprase (idursulfase)
            '540920700'     -- Elaprase (idursulfase)
        )
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        -- J-codes and procedure codes
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN (
            '99601',   -- Home infusion/specialty drug administration
            '99602',   -- Home infusion/specialty drug administration
            '96365',   -- Intravenous infusion, initial
            '96366',   -- Intravenous infusion, additional hour
            'J1743',   -- Elaprase (idursulfase) injection
            'S9357',   -- Home infusion therapy
            'S9379',   -- Home infusion therapy
            '38206',   -- Blood-derived hematopoietic progenitor cell harvesting for transplantation, per collection; autologous
            '38230',   -- Bone marrow harvesting for transplantation; allogeneic
            '38232',   -- Bone marrow harvesting for transplantation; autologous
            '38240',   -- Hematopoietic progenitor cell (HPC); allogeneic transplantation per donor
            '38241',   -- Hematopoietic progenitor cell (HPC); autologous transplantation
            '38242',   -- Allogeneic lymphocyte infusions
            '38243',   -- Hematopoietic progenitor cell (HPC); HPC boost (allogeneic)
            '38250'    -- Bone marrow or blood-derived peripheral stem cell transplantation
        )
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    -- Get Elaprase-only treatment claims (for unspecified incremental logic)
    SELECT DISTINCT PATIENT_ID
    FROM (
        -- Elaprase NDCs from medical events
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN (
            '54092070001',  -- Elaprase (idursulfase)
            '540920700'     -- Elaprase (idursulfase)
        )
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        -- Elaprase NDCs from pharmacy
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN (
            '54092070001',  -- Elaprase (idursulfase)
            '540920700'     -- Elaprase (idursulfase)
        )
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
        
        UNION ALL
        
        -- J1743 only
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'  -- Elaprase (idursulfase) injection
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    -- E76.1 patients with 2+ Dx AND treatment
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t ON p.PATIENT_ID = t.PATIENT_ID
),
Patients_Incremental_Unspecified AS (
    -- E76.3 patients with 2+ Dx, Elaprase treatment, NOT already in specified group
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t ON p.PATIENT_ID = t.PATIENT_ID
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
)
-- Final result: Specified + Incremental Unspecified
SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
UNION
SELECT PATIENT_ID FROM Patients_Incremental_Unspecified;